In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import skimage
from skimage.util import compare_images
from skimage import io, color, measure
from skimage.measure import regionprops
from skimage.measure import label as sk_label  # Renaming the label function to avoid conflicts

import matplotlib.pyplot as plt
import matplotlib

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Import images - ADJUST relative path names
# Please provide the path to the runs directory

mask_cell_runs_dir = Path(r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20231218/D56/500ms/ROIs/Whole_cell/ROIs_as_mask_BIOP')
mask_cell_dir = mask_cell_runs_dir
mask_cell_path = os.path.join(mask_cell_dir,'*.tif') 
mask_cell_files = np.sort(glob.glob(mask_cell_path))

mask_nucleus_runs_dir = Path(r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20231218/D56/500ms/ROIs/Nucleus/ROIs_as_mask_BIOP')
mask_nucleus_dir = mask_nucleus_runs_dir
mask_nucleus_path = os.path.join(mask_nucleus_dir,'*.tif') 
mask_nucleus_files = np.sort(glob.glob(mask_nucleus_path))

In [ ]:
# Read images into list

mask_cell = []
mask_nucleus = []

for file in mask_cell_files:
    image = imread(file)
    mask_cell.append(image)

for file in mask_nucleus_files:
    image = imread(file)
    mask_nucleus.append(image)

In [ ]:
np.unique(mask_cell[0])

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(mask_cell[0])
ax[1].imshow(mask_nucleus[0])

In [ ]:
# To match intensity values (= labels) of ROIs in nucleus image to intensity values of ROIs in whole cell image

for nucleus_image, cell_image in zip(mask_nucleus, mask_cell):
    # Create a dictionary to store coordinates and corresponding intensity values from mask_cell
    coordinates_intensity = {}

    # Store coordinates and intensity values from cell_image in the dictionary
    for cell_label in regionprops(cell_image):
        for coord in cell_label.coords:
            coordinates_intensity[tuple(coord)] = cell_image[coord[0], coord[1]]

    # Update nucleus_image based on the mapped coordinates and intensity values
    for nucleus_label in regionprops(nucleus_image):
        for coord in nucleus_label.coords:
            if tuple(coord) in coordinates_intensity:
                nucleus_image[coord[0], coord[1]] = coordinates_intensity[tuple(coord)]

In [ ]:
# Plot results of label matching

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(mask_cell[0])
ax[1].imshow(mask_nucleus[0])

In [ ]:
# Substract nucleus mask from cell mask and store as binary cytoplasm mask

mask_cytoplasm = []

for cell, nucleus in zip (mask_cell, mask_nucleus):
    image = skimage.util.compare_images(cell, nucleus)
    mask_cytoplasm.append(image)

In [ ]:
# Plot binary cytoplasm mask

plt.imshow(mask_cytoplasm[0])

In [ ]:
np.unique(mask_cytoplasm[0])

In [ ]:
# Converting ROI values to integers

mask_cytoplasm_int = []

for element in mask_cytoplasm:
    
    # Find unique values
    unique_values = np.unique(element)
    #print(unique_values)

    # Create a mapping dictionary
    mapping = {value: index for index, value in enumerate(unique_values)}

    # Replace values with integers
    int_array = np.array([[mapping[value] for value in row] for row in element])
    int_array = int_array.astype(float)
    
    mask_cytoplasm_int.append(int_array)
    

np.unique(mask_cytoplasm_int[0])

In [ ]:
plt.imshow(mask_cytoplasm_int[2])

In [ ]:
np.unique(mask_cytoplasm_int[2])
print(type(mask_cytoplasm[2]))
print(type(mask_cytoplasm_int[2]))

In [ ]:
# Output directory max projections - ADJUST relative path names
# Please provide the path to the output directory

mask_cytoplasm_output_dir = Path(r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20231218/D56/500ms/ROIs/Cytoplasm')
mask_cytoplasm_dir = mask_cytoplasm_output_dir

In [ ]:
# Save mask images

for img, tiff_file in zip(mask_cytoplasm_int, mask_cell_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(mask_cytoplasm_dir, file_name + '.tif')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img)